# Group I Recombination SHAP Blob Analysis

Characterizes two distinct clusters ("blobs") of genes within the Group I maize
model's negative recombination-rate SHAP values (see Fig. 3B and accompanying
Results text). Blob 1 and Blob 2 differ in their raw feature values despite
both being drawn from maize1 genes in chromosomal arms, suggesting the SHAP
clustering reflects broader chromatin/genomic context rather than recombination
rate alone.

**Blob definitions** (Group I recombination rate SHAP values):
- Blob 1: -0.80 <= recomb SHAP <= -0.40  (strong push toward maize1)
- Blob 2: -0.40 <  recomb SHAP <   0.00  (weak push toward maize1)

Genes with recomb SHAP < -0.80 (sparse tail) or >= 0 (positive SHAP, a
separate question) are excluded from both blobs.

**Steps:**
1. Reconstruct the exact StratifiedKFold fold order used during model
   training, to align the raw feature CSV with the SHAP value array, and
   define blob membership.
2. Fisher's exact test — do the two blobs differ in subgenome composition?
3. Mann-Whitney U feature comparison — do the two blobs differ in raw
   feature values? (All genes, and Maize1-only to control for subgenome
   composition.)
4. Apply manuscript display names to the Step 3 output for readability.

**Outputs** (written to `maize_outputs/blob_analysis/`, kept local and
regenerable — not tracked in git, see `.gitignore`; corresponds to **Data S3**):
- `blob_mwu_all_genes.csv`, `blob_mwu_all_genes_renamed.csv`
- `blob_mwu_maize1_only.csv`, `blob_mwu_maize1_only_renamed.csv`

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from scipy.stats import fisher_exact, mannwhitneyu, false_discovery_control

# --- Paths ---
FEATURESETS_DIR = Path("../final_featuresets")
MODEL_OUTPUTS_DIR = Path("../maize_outputs/model_outputs")

CSV_PATH = FEATURESETS_DIR / "processedMaize_groupI_all_columns_final.csv"
NPZ_PATH = MODEL_OUTPUTS_DIR / "groupI_shap_by_label.npz"

# Kept local and regenerable — not tracked in git (see .gitignore).
OUTPUT_DIR = Path("../maize_outputs/blob_analysis")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_A = OUTPUT_DIR / "blob_mwu_all_genes.csv"
OUT_B = OUTPUT_DIR / "blob_mwu_maize1_only.csv"
OUT_A_RENAMED = OUTPUT_DIR / "blob_mwu_all_genes_renamed.csv"
OUT_B_RENAMED = OUTPUT_DIR / "blob_mwu_maize1_only_renamed.csv"

print(f"CSV path    : {CSV_PATH}")
print(f"NPZ path    : {NPZ_PATH}")
print(f"Output dir  : {OUTPUT_DIR}")

CSV path    : ../final_featuresets/processedMaize_groupI_all_columns_final.csv
NPZ path    : ../maize_outputs/model_outputs/groupI_shap_by_label.npz
Output dir  : ../maize_outputs/blob_analysis


## Shared Setup: Reconstruct Fold Order and Define Blobs

Reconstructs the StratifiedKFold test-fold order used during model training
to align the raw feature matrix (CSV) with the SHAP value array (npz). Both
arrays are in cross-validation fold order, not original CSV row order.

In [4]:
# --- Load raw feature CSV ---
df = pd.read_csv(CSV_PATH)
print(f"CSV shape: {df.shape}")

# --- Replicate process_data() for Group I ---
# Exclude WGD (label), location (dropped for Group I per USE_LOCATION=False),
# Maize (gene ID), and group (metadata) — replicates the feature matrix
# passed to StratifiedKFold during model training.
EXCLUDE_COLS = ('WGD', 'location', 'Maize', 'group')
feature_names = [col for col in df.columns if col not in EXCLUDE_COLS]
X = df[feature_names].to_numpy()
y = df['WGD'].to_numpy()

assert X.shape == (4292, 59), f"CRITICAL: Unexpected X shape {X.shape}"
assert len(feature_names) == 59, f"CRITICAL: Unexpected feature count {len(feature_names)}"
print(f"X shape: {X.shape} | y shape: {y.shape}")

# --- Reconstruct StratifiedKFold fold order ---
# Must use identical parameters to the original model run.
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
fold_order_indices = []
for fold_index, (train_index, test_index) in enumerate(skf.split(X, y)):
    fold_order_indices.append(test_index)

fold_order_indices = np.concatenate(fold_order_indices, axis=0)
assert len(fold_order_indices) == 4292, "CRITICAL: fold_order_indices length mismatch"
assert len(np.unique(fold_order_indices)) == 4292, "CRITICAL: duplicate indices in fold order"

# --- Reorder dataframe to match SHAP array row order ---
df_aligned = df.iloc[fold_order_indices].reset_index(drop=False)
# 'index' column now holds the original CSV row index for traceability

# --- Load npz ---
npz = np.load(NPZ_PATH, allow_pickle=True)
shap_all    = npz['shap_all']        # shape (4292, 59)
true_labels = npz['true_labels']     # shape (4292,)
shap_feature_names = list(npz['feature_names'])

# --- Alignment sanity check ---
# WGD in reordered CSV must match true_labels from npz row-for-row.
wgd_aligned = df_aligned['WGD'].to_numpy()
assert np.all(wgd_aligned == true_labels), \
    "CRITICAL: WGD alignment failed — fold order does not match npz true_labels"
print("Alignment check passed: WGD matches true_labels across all 4292 rows")

# --- Extract recombination rate SHAP column ---
recomb_idx = shap_feature_names.index('recomb')
recomb_shap = shap_all[:, recomb_idx]

# --- Define blob membership ---
blob1_mask = (recomb_shap >= -0.80) & (recomb_shap <= -0.40)
blob2_mask = (recomb_shap >  -0.40) & (recomb_shap <   0.00)

print(f"\nBlob 1 (-0.80 to -0.40): {blob1_mask.sum()} genes")
print(f"Blob 2 (-0.40 to  0.00): {blob2_mask.sum()} genes")
print(f"Excluded (tail < -0.80): {(recomb_shap < -0.80).sum()} genes")
print(f"Excluded (SHAP >= 0)   : {(recomb_shap >= 0).sum()} genes")

# --- Subgenome composition per blob ---
blob1_labels = true_labels[blob1_mask]
blob2_labels = true_labels[blob2_mask]

print("\nSubgenome composition:")
print(f"  Blob 1 — Maize1 (WGD=0): {(blob1_labels == 0).sum()} | "
      f"Maize2 (WGD=1): {(blob1_labels == 1).sum()}")
print(f"  Blob 2 — Maize1 (WGD=0): {(blob2_labels == 0).sum()} | "
      f"Maize2 (WGD=1): {(blob2_labels == 1).sum()}")

# --- Subset aligned dataframe by blob ---
df_blob1 = df_aligned[blob1_mask].copy()
df_blob2 = df_aligned[blob2_mask].copy()

print(f"\ndf_blob1 shape: {df_blob1.shape}")
print(f"df_blob2 shape: {df_blob2.shape}")

CSV shape: (4292, 63)
X shape: (4292, 59) | y shape: (4292,)
Alignment check passed: WGD matches true_labels across all 4292 rows

Blob 1 (-0.80 to -0.40): 1560 genes
Blob 2 (-0.40 to  0.00): 1172 genes
Excluded (tail < -0.80): 167 genes
Excluded (SHAP >= 0)   : 1393 genes

Subgenome composition:
  Blob 1 — Maize1 (WGD=0): 1073 | Maize2 (WGD=1): 487
  Blob 2 — Maize1 (WGD=0): 632 | Maize2 (WGD=1): 540

df_blob1 shape: (1560, 64)
df_blob2 shape: (1172, 64)


## Step 2: Fisher's Exact Test on Subgenome Composition

Tests whether blob 1 and blob 2 differ significantly in their proportion of
Maize1 (WGD=0, dominant) vs. Maize2 (WGD=1, submissive) genes. A significant
result means the two blobs are not just recombination-rate SHAP regimes —
they also differ in subgenome identity, which must be considered when
interpreting downstream feature comparisons (Step 3).

In [5]:
# --- Contingency table ---
b1_m1 = (blob1_labels == 0).sum()   # Blob 1, Maize1
b1_m2 = (blob1_labels == 1).sum()   # Blob 1, Maize2
b2_m1 = (blob2_labels == 0).sum()   # Blob 2, Maize1
b2_m2 = (blob2_labels == 1).sum()   # Blob 2, Maize2

contingency = np.array([[b1_m1, b1_m2],
                         [b2_m1, b2_m2]])

print("Contingency table (rows = blob, cols = Maize1 / Maize2):")
print(f"{'':10s} {'Maize1':>8s} {'Maize2':>8s} {'Total':>8s}")
print(f"{'Blob 1':10s} {b1_m1:8d} {b1_m2:8d} {b1_m1+b1_m2:8d}")
print(f"{'Blob 2':10s} {b2_m1:8d} {b2_m2:8d} {b2_m1+b2_m2:8d}")
print(f"{'Total':10s} {b1_m1+b2_m1:8d} {b1_m2+b2_m2:8d} {b1_m1+b1_m2+b2_m1+b2_m2:8d}")

print(f"\nBlob 1 % Maize1: {100 * b1_m1 / (b1_m1 + b1_m2):.1f}%")
print(f"Blob 2 % Maize1: {100 * b2_m1 / (b2_m1 + b2_m2):.1f}%")

# --- Fisher's exact test ---
odds_ratio, p_value = fisher_exact(contingency, alternative='two-sided')

print(f"\nFisher's exact test:")
print(f"  Odds ratio : {odds_ratio:.4f}")
print(f"  p-value    : {p_value:.2e}")

Contingency table (rows = blob, cols = Maize1 / Maize2):
             Maize1   Maize2    Total
Blob 1         1073      487     1560
Blob 2          632      540     1172
Total          1705     1027     2732

Blob 1 % Maize1: 68.8%
Blob 2 % Maize1: 53.9%

Fisher's exact test:
  Odds ratio : 1.8826
  p-value    : 2.44e-15


## Step 3: Mann-Whitney U Feature Comparison

Compares raw feature values between blob 1 and blob 2 genes using
Mann-Whitney U tests with BH-FDR correction and rank-biserial correlation
as effect size.

Two parallel comparisons:
- **A) All genes** in each blob
- **B) Maize1 (WGD=0) genes only** — controls for the subgenome composition
  difference identified in Step 2

Rank-biserial > 0: blob 1 tends to have larger feature values.
Rank-biserial < 0: blob 2 tends to have larger feature values.

In [6]:
# --- Helper: rank-biserial correlation ---
# Positive = blob 1 tends to have larger raw feature values than blob 2
# Negative = blob 2 tends to have larger raw feature values than blob 1
def rank_biserial(u_stat, n1, n2):
    return 1 - (2 * u_stat) / (n1 * n2)

def run_mwu(df_a, df_b, feature_names):
    results = []
    for feat in feature_names:
        try:
            vals_a = pd.to_numeric(df_a[feat], errors='raise').dropna().to_numpy(dtype=float)
            vals_b = pd.to_numeric(df_b[feat], errors='raise').dropna().to_numpy(dtype=float)
        except (ValueError, TypeError):
            print(f"WARNING: Skipping feature '{feat}' — could not convert to numeric")
            continue
        u_stat, p_raw = mannwhitneyu(vals_a, vals_b, alternative='two-sided')
        rbc = rank_biserial(u_stat, len(vals_a), len(vals_b))
        results.append({
            'feature'       : feat,
            'n_blob1'       : len(vals_a),
            'n_blob2'       : len(vals_b),
            'median_blob1'  : np.median(vals_a),
            'median_blob2'  : np.median(vals_b),
            'U_stat'        : u_stat,
            'p_raw'         : p_raw,
            'rank_biserial' : rbc
        })
    results_df = pd.DataFrame(results)
    p_adj = false_discovery_control(results_df['p_raw'], method='bh')
    results_df['p_adj_BH']   = p_adj
    results_df['significant'] = p_adj < 0.05
    results_df['abs_rbc'] = results_df['rank_biserial'].abs()
    results_df = results_df.sort_values('abs_rbc', ascending=False).drop(columns='abs_rbc')
    results_df = results_df.reset_index(drop=True)
    return results_df

# --- Comparison A: all genes ---
results_A = run_mwu(df_blob1, df_blob2, feature_names)
results_A.to_csv(OUT_A, index=False)

print(f"=== COMPARISON A: All genes (Blob1 n={blob1_mask.sum():,} vs Blob2 n={blob2_mask.sum():,}) ===")
print(f"Significant features (BH-FDR < 0.05): {results_A['significant'].sum()} / {len(results_A)}")
print(results_A[['feature','median_blob1','median_blob2','rank_biserial','p_adj_BH','significant']].to_string(index=False))
print(f"\nSaved: {OUT_A}")

# --- Comparison B: Maize1 only ---
blob1_m1_mask = blob1_mask & (true_labels == 0)
blob2_m1_mask = blob2_mask & (true_labels == 0)
df_blob1_m1 = df_aligned[blob1_m1_mask].copy()
df_blob2_m1 = df_aligned[blob2_m1_mask].copy()
results_B = run_mwu(df_blob1_m1, df_blob2_m1, feature_names)
results_B.to_csv(OUT_B, index=False)

print(f"\n=== COMPARISON B: Maize1 only (Blob1 n={blob1_m1_mask.sum():,} vs Blob2 n={blob2_m1_mask.sum():,}) ===")
print(f"Significant features (BH-FDR < 0.05): {results_B['significant'].sum()} / {len(results_B)}")
print(results_B[['feature','median_blob1','median_blob2','rank_biserial','p_adj_BH','significant']].to_string(index=False))
print(f"\nSaved: {OUT_B}")

=== COMPARISON A: All genes (Blob1 n=1,560 vs Blob2 n=1,172) ===
Significant features (BH-FDR < 0.05): 16 / 59
          feature  median_blob1  median_blob2  rank_biserial      p_adj_BH  significant
           recomb      1.780000      3.555000       0.728137 1.261459e-231         True
        H2AZ_down      0.048860      0.106675       0.101467  1.207920e-04         True
        acr_Ldown  15888.500000   7473.500000      -0.100928  1.207920e-04         True
     H3K4me1_down     -0.013169     -0.000643       0.091144  5.710131e-04         True
     H3K4me3_down     -0.025685     -0.015017       0.090701  5.710131e-04         True
     CHG_down_avg      0.410000      0.325000      -0.088827  6.777749e-04         True
     H3K56ac_down     -0.000869      0.010732       0.079672  2.707505e-03         True
     H3K27ac_down      0.064138      0.090530       0.078481  2.873888e-03         True
      CG_down_avg      0.585000      0.532500      -0.074764  4.778485e-03         True
     CHG_

## Step 4: Apply Manuscript Display Names

Applies the manuscript feature display-name dictionary (matching Figs. 3–7
and the correlation/network notebooks) to the `feature` column in both
Step 3 output files.

In [7]:
feature_rename_dict = {
    'location':             'Loc',
    'GC_genic':             'GCd',
    'GC_prom':              'GCp',
    'avg_expression':       'Exp',
    'tau':                  '\u03C4',
    'Ka':                   'Ka',
    'Ks':                   'Ks',
    'O':                    '\u03C9',
    'acr_Lup':              'ACRdu',
    'acr_Ldown':            'ACRdd',
    'acr_S':                'ACRs',
    'recomb':               'Re',
    'CHH_up_avg':           'M1',
    'CHH_down_avg':         'M2',
    'CHH_body_avg':         'M3',
    'CHG_up_avg':           'M4',
    'CHG_down_avg':         'M5',
    'CHG_body_avg':         'M6',
    'CG_up_avg':            'M7',
    'CG_down_avg':          'M8',
    'CG_body_avg':          'M9',
    'CHH_up_max':           'M10',
    'CHH_down_max':         'M11',
    'CHG_up_max':           'M12',
    'CHG_down_max':         'M13',
    'CG_up_max':            'M14',
    'CG_down_max':          'M15',
    'TEdist':               'TEdi',
    'TEdenseAvgUp':         'TEu1',
    'TEdenseAvgDown':       'TEd1',
    'TEdenseMaxUp':         'TEu2',
    'TEdenseMaxDown':       'TEd2',
    'TE1':                  'TE1',
    'TE2':                  'TE2',
    'TE3':                  'TE3',
    'TE4':                  'TE4',
    'H2AZ_down':            'Hd1',
    'H3K4me1_down':         'Hd2',
    'H3K4me3_down':         'Hd3',
    'H3K9ac_down':          'Hd4',
    'H3K27ac_down':         'Hd5',
    'H3K27me3_down':        'Hd6',
    'H3K36me3_down':        'Hd7',
    'H3K56ac_down':         'Hd8',
    'H2AZ_genebody':        'Hg1',
    'H3K4me1_genebody':     'Hg2',
    'H3K4me3_genebody':     'Hg3',
    'H3K9ac_genebody':      'Hg4',
    'H3K27ac_genebody':     'Hg5',
    'H3K27me3_genebody':    'Hg6',
    'H3K36me3_genebody':    'Hg7',
    'H3K56ac_genebody':     'Hg8',
    'H2AZ_up':              'Hu1',
    'H3K4me1_up':           'Hu2',
    'H3K4me3_up':           'Hu3',
    'H3K9ac_up':            'Hu4',
    'H3K27ac_up':           'Hu5',
    'H3K27me3_up':          'Hu6',
    'H3K36me3_up':          'Hu7',
    'H3K56ac_up':           'Hu8',
}

def rename_features(in_path, out_path):
    df = pd.read_csv(in_path)

    # Check for any feature names not in the dictionary
    unmatched = set(df['feature']) - set(feature_rename_dict.keys())
    if unmatched:
        print(f"WARNING: The following features in {in_path} have no rename entry: {unmatched}")

    df['feature'] = df['feature'].map(feature_rename_dict).fillna(df['feature'])
    df.to_csv(out_path, index=False)
    print(f"Saved: {out_path} ({len(df)} rows)")

rename_features(OUT_A, OUT_A_RENAMED)
rename_features(OUT_B, OUT_B_RENAMED)

Saved: ../maize_outputs/blob_analysis/blob_mwu_all_genes_renamed.csv (59 rows)
Saved: ../maize_outputs/blob_analysis/blob_mwu_maize1_only_renamed.csv (59 rows)
